# Clip, Intersect and Dissolve

**Series:** Geoprocessing in Python (notebook 2 of 3)
**Time:** about 30 minutes

**In this notebook you will:**
- Clip a layer to a boundary
- Intersect two polygon layers and measure the pieces
- Estimate the population in a zone with areal weighting, and know its assumption
- Dissolve polygons by a column and sum their values

**You should already know:** notebook 01a (buffers and select by location).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## Overlay operations

Buffers and selections answer "which features". **Overlays** cut shapes against each other and create new
ones:

| Operation | Result | GeoPandas |
|---|---|---|
| **Clip** | The parts of layer A inside layer B; B's attributes are not added | `gpd.clip(a, b)` |
| **Intersect** | The parts where A and B overlap, with the attributes of both | `gpd.overlay(a, b, how="intersection")` |
| **Dissolve** | Merges features, optionally by a column, and summarizes values | `a.dissolve(by="col", aggfunc="sum")` |

We start where notebook 01a ended: tracts with ACS data, supermarkets and the 1-mile zone.

In [ ]:
import geopandas as gpd
import pandas as pd

stores = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/grocery_stores.geojson")
city = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/study_area.geojson")
tracts = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_tracts.geojson")
acs = pd.read_csv("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_acs_2024.csv", dtype={"GEOID": str})

# Work in meters: UTM zone 16N
stores = stores.to_crs(epsg=26916)
city = city.to_crs(epsg=26916)
tracts = tracts.to_crs(epsg=26916).merge(acs.drop(columns="name"), on="GEOID")
supermarkets = stores[stores["store_type"] == "Supermarket"]

near = supermarkets.copy()
near["geometry"] = supermarkets.buffer(1609)
near = near.dissolve()[["geometry"]]
near["zone"] = "within 1 mile"
print(len(tracts), "tracts,", len(supermarkets), "supermarkets")

## Step 1: Clip the tracts to the city

The grocery store layer only covers the two cities, so we limit the analysis to the city limits. Outside them
we would be missing stores (an **edge effect**) and rural tracts would look worse off than they are.

In [ ]:
tracts["tract_m2"] = tracts.area             # area of the whole tract, before clipping
in_city = gpd.clip(tracts, city)
in_city["share_in_city"] = in_city.area / in_city["tract_m2"]

print(len(in_city), "tracts reach into the city")
in_city[["name", "share_in_city"]].sort_values("share_in_city").head(8).round(2)

Many tracts are only partly inside the city. To estimate the city's population we need to split each tract's
count. The simplest method is **areal weighting**: if 40% of a tract's area is in the city, assume 40% of its
people are.

In [ ]:
in_city["pop_in_city"] = in_city["share_in_city"] * in_city["total_pop"]
print("Estimated city population:", round(in_city["pop_in_city"].sum()))

The 2020 Census counted about 115,000 people in the two cities, so the estimate is close. Areal weighting
assumes people are spread **evenly** across a tract. That is rarely true (farmland and parks have no homes), so
it works best when the pieces are large or the tracts are mostly urban. Better methods weight by smaller units
such as blocks, or by land cover (**dasymetric** mapping).

## Step 2: Intersect with the 1-mile zone

`overlay(..., how="intersection")` keeps only the overlapping parts and gives each piece the columns from both
layers.

In [ ]:
pieces = gpd.overlay(in_city, near, how="intersection")
pieces["pop_near"] = pieces.area / pieces["tract_m2"] * pieces["total_pop"]

near_pop = pieces["pop_near"].sum()
city_pop = in_city["pop_in_city"].sum()
print(f"About {near_pop:,.0f} of {city_pop:,.0f} residents ({near_pop / city_pop:.0%}) live within 1 mile of a supermarket")

In [ ]:
ax = in_city.plot(color="whitesmoke", edgecolor="lightgray", figsize=(7, 7))
pieces.plot(ax=ax, column="pop_near", cmap="Greens", edgecolor="white", legend=True,
            legend_kwds={"label": "Estimated residents in the piece", "shrink": 0.6})
ax.set_title("Pieces of tracts within 1 mile of a supermarket")
ax.set_axis_off()

## Step 3: Dissolve by a column

`overlay(how="union")` would keep both the near and the far pieces. A simpler route is `difference`: the parts
of the city tracts **outside** the zone. We label both sets and dissolve by the label, summing the estimated
households without a car.

In [ ]:
far = gpd.overlay(in_city, near, how="difference")
far["zone"] = "more than 1 mile"
both = pd.concat([pieces, far], ignore_index=True)
both["nocar_est"] = both.area / both["tract_m2"] * both["hh_no_vehicle"]

summary = both.dissolve(by="zone", aggfunc={"nocar_est": "sum"})
summary["area_km2"] = summary.area / 1e6
summary[["nocar_est", "area_km2"]].round(1)

`dissolve(by=...)` merges every piece with the same value into one feature, and `aggfunc` says how to combine
the numbers (`sum`, `mean`, `max` and so on). Summing makes sense for **counts** (people, households). Never sum
rates or medians such as `median_income`; recompute them from counts instead.

## Try it yourself

Estimate how many **children (under 18)** in the city live more than 1 mile from a supermarket.

Use the `both` layer and the `pop_under18` column with the same areal weighting.

*Hint:* Copy the `nocar_est` line and change the column.

<details>
<summary><b>Show a solution</b></summary>

```python
both["kids_est"] = both.area / both["tract_m2"] * both["pop_under18"]
both.groupby("zone")["kids_est"].sum().round()
```

</details>

In [ ]:
# Write your answer here

## Recap

- `gpd.clip()` cuts a layer to a boundary; `overlay()` intersects (or differences) two polygon layers
- Keep the original area before cutting, so you can work out each piece's share
- Areal weighting splits counts by area share and assumes people are spread evenly
- `dissolve(by=..., aggfunc='sum')` merges features and adds up counts; never sum rates or medians

**Next:** [01c Distance and Nearest Features](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/01c_Distance_and_Nearest_Features.ipynb)